In [ ]:
"""
=============================================================================
PROJETO: Classificador de Phishing e Engenharia Social (Fase 2)
=============================================================================

* INTENÇÃO DA MUDANÇA (TEXT DROPOUT E CURA DE ATALHOS):
Este notebook foca em resolver o problema de "Shortcut Learning" descoberto
na Fase 1. A intenção foi retirar a dependência do modelo em relação aos
links maliciosos explícitos, obrigando a rede neural a interpretar as
ferramentas psicológicas do golpe (chantagem, medo, falsa urgência, lucro
fácil) utilizando puramente as palavras da mensagem.

* DADOS UTILIZADOS:
- O dataset da fase anterior ('meu_super_dataset_phishing_v2.csv'), que foi
  submetido a uma limpeza via Regex para padronizar todos os links reais[cite: 3].
- Remoção aleatória da tag '[URL]' em 50% dos casos de fraude (resultando em
  1.872 tags removidas) para criar variabilidade de treinamento[cite: 3].
- Dataset consolidado salvo como 'dataset_filtrando_URL.csv', gerando um
  treinamento balanceado com 10.566 mensagens[cite: 3].

* RESULTADOS PRELIMINARES:
Ao retirar a muleta estrutural, o modelo sofreu uma queda de performance
esperada no teste 'Out of Bounds' (Acurácia de 76% e Recall de 52% contra
phishing)[cite: 3]. O diagnóstico revelou a criação de um "Trigger Bias"
(Viés de Gatilho). O modelo se tornou paranoico com palavras financeiras e
passou a bloquear conversas normais apenas por conterem palavras isoladas
como "Pix", "boleto", "fatura" ou "Magalu".

* POSSÍVEIS MELHORIAS (PRÓXIMO PASSO):
Curar a paranoia do modelo através da injeção de "Hard Negatives".
Precisamos criar um conjunto de dados contendo conversas rotineiras e
inofensivas que utilizem propositalmente esses gatilhos financeiros,
ensinando a IA a avaliar as combinações de palavras em vez de penalizar
termos isolados.
=============================================================================
"""

In [ ]:
# 1. Instalar a NOVA biblioteca oficial do Google
!pip install -q google-genai pandas
!pip install -q datasets deep-translator pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.0 MB/s eta 0:00:00


In [ ]:
# 1. Instalar as bibliotecas necessárias
import json
import time
from google import genai
import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset, Dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch
#!git clone -b desenv https://github.com/jmmfilho/sec-fraudimabr.git

In [ ]:
import pandas as pd
import numpy as np
import re

print("Carregando o dataset V2...")
df = pd.read_csv('meu_super_dataset_phishing_v2.csv')

# ==========================================
# PASSO 1: PADRONIZAR TODOS OS LINKS REAIS
# ==========================================
# Regex avançada para capturar http, https, www, e domínios comuns no meio do texto
url_pattern = r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))'

def padronizar_links(texto):
    if pd.isna(texto):
        return texto

    # Substitui os links encontrados pela tag [URL]
    texto_padronizado = re.sub(url_pattern, '[URL]', str(texto))

    # Limpa possíveis duplicações seguidas (ex: se havia dois links juntos, vira um só [URL])
    texto_padronizado = re.sub(r'(\[URL\]\s*)+', '[URL] ', texto_padronizado)

    return texto_padronizado.strip()

print("Convertendo links reais para a tag [URL]...")
df['texto'] = df['texto'].apply(padronizar_links)

# Verificação intermediária
qtd_com_url = df['texto'].str.contains(r'\[URL\]', na=False, regex=True).sum()
print(f"Total de mensagens com [URL] após padronização: {qtd_com_url}")


# ==========================================
# PASSO 2: APLICAR O TEXT DROPOUT (50%)
# ==========================================
# Fixar a seed para garantir resultados reproduzíveis
np.random.seed(42)

def ocultar_url_aleatoriamente(texto):
    if pd.isna(texto):
        return texto

    if '[URL]' in str(texto):
        # 50% de chance de remover a tag
        if np.random.rand() > 0.5:
            return str(texto).replace('[URL]', '').strip()

    return texto

print("Aplicando o Text Dropout (removendo 50% das tags)...")
df['texto'] = df['texto'].apply(ocultar_url_aleatoriamente)

# Verificação final
qtd_final = df['texto'].str.contains(r'\[URL\]', na=False, regex=True).sum()
print(f"Total de mensagens com [URL] após o Dropout: {qtd_final}")
print(f"Tags removidas para forçar o aprendizado: {qtd_com_url - qtd_final}")

# ==========================================
# PASSO 3: SALVAR A VERSÃO FINAL
# ==========================================
df.to_csv('dataset_filtrando_URL.csv', index=False)
print("\n✅ Dataset V3 higienizado e salvo como 'dataset_filtrando_URL.csv'!")

Carregando o dataset V2...
Convertendo links reais para a tag [URL]...
Total de mensagens com [URL] após padronização: 3717
Aplicando o Text Dropout (removendo 50% das tags)...
Total de mensagens com [URL] após o Dropout: 1845
Tags removidas para forçar o aprendizado: 1872

✅ Dataset V3 higienizado e salvo como 'dataset_filtrando_URL.csv'!


In [ ]:


print("Iniciando a preparação dos dados...")

# ==========================================
# PASSO 1: PREPARAÇÃO E BALANCEAMENTO
# ==========================================

# 1. Carregar o novo Super Dataset
df = pd.read_csv("dataset_filtrando_URL.csv")
df = df.dropna(subset=['texto'])

# Força toda a coluna 'texto' a ser do tipo string (texto)
df['texto'] = df['texto'].astype(str)
# 2. Balanceamento Estrito (Undersampling)
# Descobre qual é a classe com menos exemplos e iguala a outra para ter o mesmo tamanho
tamanho_menor_classe = df['label'].value_counts().min()

df_normal = df[df['label'] == 0].sample(tamanho_menor_classe, random_state=42)
df_golpe = df[df['label'] == 1].sample(tamanho_menor_classe, random_state=42)

# Junta tudo e embaralha
df_balanceado = pd.concat([df_normal, df_golpe]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset balanceado com sucesso! Total: {len(df_balanceado)} mensagens.")
print(df_balanceado['label'].value_counts())

# 3. Divisão de Treino (80%) e Validação (20%)
df_train, df_val = train_test_split(df_balanceado, test_size=0.3, random_state=42)

# As colunas já estão como 'texto' e 'label', vamos renomear para o padrão Hugging Face
df_train.columns = ['text', 'labels']
df_val.columns = ['text', 'labels']

# Converter Pandas para Dataset da Hugging Face
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val, preserve_index=False)

# ==========================================
# PASSO 2: TREINAMENTO (API MODEL)
# ==========================================

# 1. Inicializar Tokenizer e Modelo
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Função de Tokenização
def tokenizar_funcao(exemplos):
    return tokenizer(exemplos["text"], padding=False, truncation=True, max_length=128)

print("Tokenizando os textos...")
tokenized_train = train_dataset.map(tokenizar_funcao, batched=True)
tokenized_val = val_dataset.map(tokenizar_funcao, batched=True)

# 3. Configurar Data Collator (Otimiza a memória da GPU preenchendo os textos dinamicamente)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Configurar Hiperparâmetros
training_args = TrainingArguments(
    output_dir="./resultados_bert_phishing",
    eval_strategy="epoch",             # Avalia no fim de cada época
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4 ,               # 3 épocas é o padrão ouro para BERT
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# 5. Inicializar o Treinador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,        # Parâmetro atualizado (substitui o antigo 'tokenizer')
    data_collator=data_collator
)

# 6. INICIAR O TREINAMENTO
print("Iniciando o treinamento na GPU...")
trainer.train()

# 7. Salvar o modelo final
model.save_pretrained("./meu_modelo_api_phishing")
tokenizer.save_pretrained("./meu_modelo_api_phishing")
print("✅ Modelo salvo com sucesso na pasta './meu_modelo_api_phishing'!")

Iniciando a preparação dos dados...
Dataset balanceado com sucesso! Total: 10566 mensagens.
label
0    5283
1    5283
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Tokenizando os textos...


Map:   0%|          | 0/7396 [00:00<?, ? examples/s]

Map:   0%|          | 0/3170 [00:00<?, ? examples/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo salvo com sucesso na pasta './meu_modelo_api_phishing'!


In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
# Carregar o pipeline de classificação com o modelo que você acabou de treinar
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1 # Usa GPU se disponível
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
print("Carregando o modelo salvo...")
# 1. Carregar o modelo treinado diretamente da pasta onde foi salvo
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1, # Usa GPU para acelerar a validação
    truncation=True,    # Corta textos que forem maiores que o limite
    max_length=128
)

# 2. Extrair os textos e os rótulos (labels) do dataset de validação (df_val)
# Nota: Assumimos que a variável df_val ainda está na memória do Colab.
textos_val = df_val['text'].tolist()
y_true_val = df_val['labels'].tolist()

print(f"Realizando predições em {len(textos_val)} mensagens de validação. Isso pode levar alguns segundos...")
# 3. Fazer as predições em lote
resultados_val = classificador(textos_val)

# 4. Extrair os valores preditos (0 ou 1) da resposta do modelo
y_pred_val = []
for resultado in resultados_val:
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Extraímos apenas o número.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred_val.append(label_previsto)

# 5. Calcular as métricas
acuracia_val = accuracy_score(y_true_val, y_pred_val)
f1_val = f1_score(y_true_val, y_pred_val, average='binary')

# 6. Exibir o Relatório Completo
print("\n" + "="*50)
print(" RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)")
print("="*50)
print(f"Acurácia Global: {acuracia_val:.4f} ({acuracia_val * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1_val:.4f} ({f1_val * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true_val, y_pred_val, target_names=["Normal (0)", "Phishing (1)"]))

Carregando o modelo salvo...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Realizando predições em 3170 mensagens de validação. Isso pode levar alguns segundos...

 RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)
Acurácia Global: 0.9956 (99.56%)
F1-Score (Fraudes): 0.9956 (99.56%)

Relatório Detalhado:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00      1584
Phishing (1)       1.00      1.00      1.00      1586

    accuracy                           1.00      3170
   macro avg       1.00      1.00      1.00      3170
weighted avg       1.00      1.00      1.00      3170



In [ ]:
# 1. Modificando as mensagens para incluir o rótulo real (Ground Truth)
# Formato: ("Texto da mensagem", rótulo_real)
# Assumindo que 0 = Normal e 1 = Phishing (verifique a ordem do seu CSV)
mensagens_teste = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("Oi, não esquece de passar na padaria voltando do trabalho.", 0),
    ("A reunião foi transferida para a sala 3, avisem os demais.", 0),
    ("Você viu o jogo ontem? Que golaço no final!", 0),
    ("Seu pedido Magalu 4920 foi entregue ao destinatário.", 0),
    ("Feliz Natal, família! Que a noite de vocês seja iluminada.", 0),
    ("Preciso que você me envie o PDF daquela apresentação até meio-dia.", 0),
    ("Estou no trânsito, devo atrasar uns 10 minutinhos.", 0),
    ("Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.", 0),
    ("Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.", 0),
    ("Bora marcar um futebol no fim de semana?", 0),
    ("Amor, a ração do cachorro acabou, compra mais depois.", 0),
    ("A sua fatura da Vivo com vencimento em 10/08 está disponível no app.", 0),
    ("Mãe, cheguei bem no hotel. Mais tarde te ligo.", 0),
    ("Alguém tem o contato de um bom encanador pra me passar?", 0),
    ("A professora de inglês mandou o link da aula no grupo.", 0),
    ("Oi, desculpa a demora pra responder, tava em reunião.", 0),
    ("Você vai almoçar em casa hoje ou no restaurante?", 0),
    ("Parabéns, amigo! Muitas felicidades e sucesso sempre.", 0),
    ("Uber: Seu motorista Carlos está chegando em 2 minutos.", 0),
    ("O boleto do condomínio já foi enviado pro seu email.", 0),
    ("Vou viajar no feriado, depois a gente se vê.", 0),
    ("Tem pão fresco saindo agora, vai querer que eu leve?", 0),
    ("A vacinação da gripe começou no posto aqui da esquina.", 0),
    ("Não consegui falar com você, me liga quando puder.", 0),
    ("Confirmando o nosso jantar de amanhã às 20h. Até lá!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("BANCO CENTRAL: Há um Pix de R$ 850 retido em seu nome. Acesse para liberar: ", 1),
    ("C6 Bank: Seu cartão foi bloqueado por segurança. Valide sua identidade imediatamente: ", 1),
    ("Mãe, meu celular quebrou, salva esse meu número novo provisório.", 1),
    ("Olá, somos do RH da Amazon. Temos vaga de meio período para ganhar de 200 a 500 por dia. Clique: ", 1),
    ("Detran informa: Suspensão de CNH agendada. Pague o débito para evitar o cancelamento: ", 1),
    ("Aviso gov.br: Sua conta será desativada em 2h. Atualize seus dados cadastrais em ", 1),
    ("CORREIOS: Seu pedido foi retido na alfândega. Taxa de R$ 22,50 pendente. Pague aqui: ", 1),
    ("Netflix: O pagamento do mês falhou. Atualize seu cartão de crédito para não perder o acesso: ", 1),
    ("Promoção O Boticário! Responda o questionário e ganhe um kit grátis de Dia das Mães: ", 1),
    ("Itaú: Identificamos um acesso no Rio de Janeiro. Caso não tenha sido você, conteste em [URL]", 1),
    ("Você tem 20.000 pontos Livelo expirando hoje! Troque por milhas ou produtos em ", 1),
    ("Serasa Consumidor: Pague sua dívida com 95% de desconto no Feirão Limpa Nome. Pix oficial: ", 1),
    ("O Governo aprovou um Auxílio Gás de R$ 120. Consulte se seu CPF tem direito: [URL]", 1),
    ("Oi, encontrei essas fotos suas da festa de ontem rolando na internet, dá uma olhada [URL] ", 1),
    ("Shopee: Parabéns, você foi selecionado para avaliar produtos. Renda diária de R$ 300. Chame no WhatsApp: ", 1),
    ("Nubank: Sua compra no valor de R$ 2.500 no MercadoPago foi aprovada. Para cancelar, clique: ", 1),
    ("Aviso Receita Federal: Sua declaração de imposto de renda caiu na malha fina. Veja a divergência: ", 1),
    ("WhatsApp: Insira o código de 6 dígitos que você recebeu por SMS para não perder sua conta.", 1),
    ("Caixa Tem: Seu saque extraordinário foi liberado. Acesse a conta para transferir o valor: ", 1),
    ("Santander: Seu aplicativo de segurança está desatualizado. Evite multas atualizando no link: ", 1),
    ("Oportunidade Única! Invista R$ 50 no nosso robô de Pix e receba R$ 500 em 10 minutos. Garanta sua vaga: ", 1),
    ("Claro: Sua fatura em aberto pode ser paga com 50% de desconto hoje no Pix. Gere a chave em: ", 1),
    ("Atenção: Ação judicial protocolada em seu CPF. Visualize o processo e mandado de intimação aqui: ", 1),
    ("Empréstimo liberado de R$ 10.000 para negativados sem consulta ao SPC! Assine o contrato online: ", 1),
    ("Mercado Livre: Seu acesso está suspenso temporariamente por atividade suspeita. Confirme sua titularidade", 1)
]

# 2. Separar os textos e os rótulos verdadeiros para processamento
textos = [item[0] for item in mensagens_teste]
y_true = [item[1] for item in mensagens_teste]

# 3. Fazer as predições usando o pipeline da Hugging Face
resultados = classificador(textos)

# 4. Extrair as predições do modelo (y_pred)
y_pred = []
print("--- DETALHAMENTO DAS MENSAGENS ---\n")
for texto, real, resultado in zip(textos, y_true, resultados):
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Pegamos apenas o número final e convertemos para inteiro.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred.append(label_previsto)

    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Certeza: {resultado['score']:.4f}\n")

# 5. Calcular e exibir as métricas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary') # 'binary' é ideal para problemas de duas classes

print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: Oi, não esquece de passar na padaria voltando do trabalho.
Real: 0 | Previsto: 0 | Certeza: 0.9982

Mensagem: A reunião foi transferida para a sala 3, avisem os demais.
Real: 0 | Previsto: 0 | Certeza: 0.9997

Mensagem: Você viu o jogo ontem? Que golaço no final!
Real: 0 | Previsto: 0 | Certeza: 0.9990

Mensagem: Seu pedido Magalu 4920 foi entregue ao destinatário.
Real: 0 | Previsto: 0 | Certeza: 0.9884

Mensagem: Feliz Natal, família! Que a noite de vocês seja iluminada.
Real: 0 | Previsto: 0 | Certeza: 0.9997

Mensagem: Preciso que você me envie o PDF daquela apresentação até meio-dia.
Real: 0 | Previsto: 0 | Certeza: 0.9995

Mensagem: Estou no trânsito, devo atrasar uns 10 minutinhos.
Real: 0 | Previsto: 0 | Certeza: 0.9990

Mensagem: Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.
Real: 0 | Previsto: 1 | Certeza: 0.9976

Mensagem: Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.
Real: 0 | Previs

In [ ]:
mensagens_teste_2 = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("oi, n esquece de passar na padaria voltando do trampo", 0),
    ("a reunião mudou pra sala 3, avisa a glr pfv", 0),
    ("vc viu o jogo ontem? q golaço no final slc!", 0),
    ("seu pacote do magalu chegou aqui na portaria, dps pega la", 0),
    ("feliz natal fml!! q a noite de vcs seja top", 0),
    ("preciso q vc me mande o pdf daquela apresentaçao ate umas 12h", 0),
    ("to no transito, vou atrasar uns 10 min malz", 0),
    ("bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm", 0),
    ("oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?", 0),
    ("bora marcar um fut esse fds?", 0),
    ("amor, a raçao do dog acabou, compra + dps", 0),
    ("a fatura da vivo ja ta no app, n esquece de pagar", 0),
    ("mae cheguei dboa no hotel. mais tarde te ligo bjs", 0),
    ("alguem tem ctt de um encanador bom pra me passar?", 0),
    ("a prof de ingles mandou o link da aula no gp", 0),
    ("oi, dsclp a demora pra responder, tava em reuniao", 0),
    ("vc vai almoçar em ksa hj ou vai comer fora?", 0),
    ("parabens mano!! mtas felicidades e sucesso smp tmj", 0),
    ("o uber ta chegando, 2 minutinhos", 0),
    ("o boleto do cond ja ta no seu email", 0),
    ("vou viajar no feriado, dps a gnt se ve", 0),
    ("tem pão fresco saindo agr, qr q eu leve?", 0),
    ("a vacina da gripe começou no posto aq da esquina", 0),
    ("n consegui falar c/ vc, me liga qnd der", 0),
    ("confirmando nosso jantar de amanhã as 20h. ate!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("Bacen: tem um pix de 850 conto retido no seu cpf. acessa p/ liberar: ", 1),
    ("C6 Bank: cartao bloqueado por seguranca. valida ai rapidao senao cancela: ", 1),
    ("Oii mae, meu cel quebrou :( salva meu num novo provisorio ai", 1),
    ("Opa, sou do RH da Amazon. Vaga meio periodo tirando de 200 a 500/dia. Clica ai: ", 1),
    ("Detran: suspensao de CNH agendada. Paga o debito p/ evitar BO no doc: ", 1),
    ("gov.br: sua conta vai ser desativada hj! atualiza seus dados aqui ", 1),
    ("Correios: seu pedido ficou retido na alfandega. Tem taxa de 22,50 pendente no pix: ", 1),
    ("Netflix: o pgto falhou :( atualiza o cartao pra n perder o acesso: ", 1),
    ("Promoção O Boticário bombando! responde a pesquisa e ganha 1 kit de dia das mães free: ", 1),
    ("Itau: acesso em outro estado detectado. Nao foi vc? Contesta aq no link [URL]", 1),
    ("vc tem 20k de pontos Livelo expirando HOJE!! troca ai por milha no link ", 1),
    ("Serasa: limpe seu nome c/ 95% off no feirao! paga no pix oficial aq: ", 1),
    ("Governo liberou Auxilio Gas de 120$!! ve se seu cpf tem direito [URL]", 1),
    ("nossa, vc viu as fotos suas da festa q vazaram?? q loucura, olha ai [URL] ", 1),
    ("Oie, vc foi selecionado p/ avaliar produtos na Shopee e tirar +300/dia. Chama no whats: ", 1),
    ("Nubank: compra de 2.500 aprovada no mercadopago. nao reconhece? clica aq pra cancelar: ", 1),
    ("Receita Federal: seu IRPF caiu na malha fina. acesse pra ver oq deu errado: ", 1),
    ("Suporte WhatsApp: digite o cod q vc recebeu por sms pra n perder o whats.", 1),
    ("Caixa Tem: saque extraordinario na conta!! clica pra transferir o dindin: ", 1),
    ("Santander: app desatualizado. atualiza no link p/ n tomar multa no cpf: ", 1),
    ("quer fzr renda extra?? investe 50tao no robo do pix e tira 500 na hora. clica: ", 1),
    ("Claro: fatura em aberto com 50% de desc hj no pix. gera a chave ai: ", 1),
    ("Atenção! açao judicial no seu CPF. ve o processo e o mandado aqui: ", 1),
    ("emprestimo de 10k aprovado ate pra negativado slc!! s/ consulta no spc. assina ai: ", 1),
    ("Mercado Livre: conta suspensa p/ ativ suspeita. confirma q é vc aqui ", 1)
]

textos = [item[0] for item in mensagens_teste_2]
y_true = [item[1] for item in mensagens_teste_2]

# 3. Fazer as predições usando o pipeline da Hugging Face
resultados = classificador(textos)

# 4. Extrair as predições do modelo (y_pred)
y_pred = []
print("--- DETALHAMENTO DAS MENSAGENS ---\n")
for texto, real, resultado in zip(textos, y_true, resultados):
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Pegamos apenas o número final e convertemos para inteiro.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred.append(label_previsto)

    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Certeza: {resultado['score']:.4f}\n")

# 5. Calcular e exibir as métricas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary') # 'binary' é ideal para problemas de duas classes

print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: oi, n esquece de passar na padaria voltando do trampo
Real: 0 | Previsto: 0 | Certeza: 0.9989

Mensagem: a reunião mudou pra sala 3, avisa a glr pfv
Real: 0 | Previsto: 0 | Certeza: 0.9948

Mensagem: vc viu o jogo ontem? q golaço no final slc!
Real: 0 | Previsto: 0 | Certeza: 0.9944

Mensagem: seu pacote do magalu chegou aqui na portaria, dps pega la
Real: 0 | Previsto: 1 | Certeza: 0.9991

Mensagem: feliz natal fml!! q a noite de vcs seja top
Real: 0 | Previsto: 0 | Certeza: 0.9986

Mensagem: preciso q vc me mande o pdf daquela apresentaçao ate umas 12h
Real: 0 | Previsto: 1 | Certeza: 0.8884

Mensagem: to no transito, vou atrasar uns 10 min malz
Real: 0 | Previsto: 0 | Certeza: 0.6759

Mensagem: bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm
Real: 0 | Previsto: 1 | Certeza: 0.9994

Mensagem: oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?
Real: 0 | Previsto: 0 | Certeza: 0.9644

Mensagem: bora marc